In [73]:
#pip install datasets


### Load Original HiTab & Statistics

Extract the baseline statistics for the original HiTab dataset, mapping unique table_id, table_source, and the sum of Question-Answer (QA) pairs per table.

In [74]:
from datasets import load_dataset
import pandas as pd

# 1. Load the original HiTab dataset directly from the Hugging Face Hub
hitab_dataset = load_dataset("kasnerz/hitab")



In [75]:
# Combine train, validation, and test splits into a single DataFrame
df_hitab_raw = pd.concat([
    hitab_dataset['train'].to_pandas(),
    hitab_dataset['validation'].to_pandas(),
    hitab_dataset['test'].to_pandas()
], ignore_index=True)

# In this Hugging Face schema, each row is a single QA pair. 
# Group by table_id and table_source to calculate total QA pairs per table.
df_hitab = df_hitab_raw.groupby(['table_id', 'table_source']).size().reset_index(name='qa_count')

print("=== 1. Original HiTab Statistics ===")
print(f"Total Unique Tables: {df_hitab['table_id'].nunique()}")
print(f"Total QA Pairs: {df_hitab['qa_count'].sum()}")
print("\nDistribution by Table Source:")
print(df_hitab.groupby('table_source')['table_id'].nunique())

=== 1. Original HiTab Statistics ===
Total Unique Tables: 3597
Total QA Pairs: 10672

Distribution by Table Source:
table_source
nsf         183
statcan    1563
totto      1851
Name: table_id, dtype: int64


In [76]:
df_hitab

,table_id,table_source,qa_count
0,0_1_nsf21326-tab001,nsf,1
1,0_totto109-3,totto,1
2,0_totto115-4,totto,1
3,0_totto74-1,totto,1
4,100,statcan,4
...,...,...,...
3592,9_totto1028-1,totto,1
3593,9_totto1092-2,totto,1
3594,9_totto1101-3,totto,1
3595,9_totto1105-4,totto,1


### Load Original IndoHiTabv1 & Statistics

Extract the baseline statistics for the original IndoHiTab dataset, mapping unique table_id, table_source, and the sum of Question-Answer (QA) pairs per table.

In [77]:


# 1. Load the original indoHiTab dataset directly from the Hugging Face Hub
indohitab_dataset = load_dataset("rizki-syazali/indohitab")


In [78]:

# Combine train, validation, and test splits into a single DataFrame
df_indohitab_raw = pd.concat([
    indohitab_dataset['train'].to_pandas(),
    indohitab_dataset['test'].to_pandas()
], ignore_index=True)

# In this Hugging Face schema, each row is a single QA pair. 
# Group by table_id and table_source to calculate total QA pairs per table.
df_indohitab = df_indohitab_raw.groupby(['table_id', 'table_source']).size().reset_index(name='qa_count')

print("=== 1. Original HiTab Statistics ===")
print(f"Total Unique Tables: {df_indohitab['table_id'].nunique()}")
print(f"Total QA Pairs: {df_indohitab['qa_count'].sum()}")
print(f"Total QA Test Pairs: {len(indohitab_dataset['test'])}")
print(f"Total QA Train Pairs: {len(indohitab_dataset['train'])}")
print("\nDistribution by Table Source:")
print(df_indohitab.groupby('table_source')['table_id'].nunique())

=== 1. Original HiTab Statistics ===
Total Unique Tables: 600
Total QA Pairs: 2880
Total QA Test Pairs: 554
Total QA Train Pairs: 2326

Distribution by Table Source:
table_source
statcan    600
Name: table_id, dtype: int64


In [79]:
# Convert to set for O(1) lookup efficiency and force string type to prevent matching errors
indohitab_ids = {str(tid) for tid in df_indohitab['table_id']}
print(f"=== 2. IndoHiTab ===")
print(f"Total Unique Tables in IndoHiTab v1: {len(indohitab_ids)}")

# 3. hitab exclude table id in indohitab -> indohitabv2_1
df_indohitabv2_1 = df_hitab[~df_hitab['table_id'].isin(indohitab_ids)].copy()
print(f"\n=== 3. IndoHiTab v2_1 ===")
print(f"Total Tables remaining after excluding v1: {df_indohitabv2_1['table_id'].nunique()}")

=== 2. IndoHiTab ===
Total Unique Tables in IndoHiTab v1: 600

=== 3. IndoHiTab v2_1 ===
Total Tables remaining after excluding v1: 2997


### Calculate Remaining QA and Exclude Tables with 0 QA (indohitabv2_2)

In [80]:
# 4. The remaining qa that did not in indohitab but in hitab
remaining_qa_v2_1 = df_indohitabv2_1['qa_count'].sum()
print(f"=== 4. Remaining QA ===")
print(f"Total QA pairs available in indohitabv2_1: {remaining_qa_v2_1}")

# 5. indohitabv2_1 -> check if tables not had qa exclude them to indohitabv2_2
df_indohitabv2_2 = df_indohitabv2_1[df_indohitabv2_1['qa_count'] > 0].copy()
tables_dropped = df_indohitabv2_1['table_id'].nunique() - df_indohitabv2_2['table_id'].nunique()

print(f"\n=== 5. IndoHiTab v2_2 ===")
print(f"Tables dropped (0 QA pairs): {tables_dropped}")
print(f"Total Valid Tables in indohitabv2_2: {df_indohitabv2_2['table_id'].nunique()}")

=== 4. Remaining QA ===
Total QA pairs available in indohitabv2_1: 7792

=== 5. IndoHiTab v2_2 ===
Tables dropped (0 QA pairs): 0
Total Valid Tables in indohitabv2_2: 2997


In [81]:
# 6. See distribution based table id unique and table source

print("=== 6.0  Distribution (Hitab v2_2) ===")
distribution_hitab = df_hitab.groupby('table_source').agg(
    unique_tables=('table_id', 'nunique'),
    total_qa=('qa_count', 'sum')
).reset_index()
print(distribution_hitab.to_string(index=False))

print("=== 6.1 Distribution (Hitab v2_1) ===")
distribution_v1 = df_indohitab.groupby('table_source').agg(
    unique_tables=('table_id', 'nunique'),
    total_qa=('qa_count', 'sum')
).reset_index()
print(distribution_v1.to_string(index=False))


print("=== 6. Final Distribution (IndoHiTab v2_2) ===")
distribution_v2_2 = df_indohitabv2_2.groupby('table_source').agg(
    unique_tables=('table_id', 'nunique'),
    total_qa=('qa_count', 'sum')
).reset_index()

print(distribution_v2_2.to_string(index=False))

=== 6.0  Distribution (Hitab v2_2) ===
table_source  unique_tables  total_qa
         nsf            183       762
     statcan           1563      7943
       totto           1851      1967
=== 6.1 Distribution (Hitab v2_1) ===
table_source  unique_tables  total_qa
     statcan            600      2880
=== 6. Final Distribution (IndoHiTab v2_2) ===
table_source  unique_tables  total_qa
         nsf            183       762
     statcan            963      5063
       totto           1851      1967


## Separate IndoHiTab v1 from HiTab

Split the data into two independent parts, each in its own variables and folder:

- **IndoHiTab v1** (Indonesian, 600 tables / 2880 QA) → `df_v1_tables`, `df_v1_qa`, saved in `data_indohitab_v1/`.
- **HiTab without v1** (original English tables not in v1, 2997 tables / 7792 QA) → `df_hitab_nonv1_tables` (with `table_content`), `df_hitab_nonv1_qa`, saved in `data_hitab_excl_v1/`.

Matching is by `table_id`. The cell verifies the parts are disjoint and together equal the full HiTab (3597 tables / 10672 QA). The v1 `table`, `answer_coordinates` and `answer_text` columns hold nested arrays, so they are stored as JSON strings.

In [82]:
import os
import json

V1_DIR = 'data_indohitab_v1'
HITAB_NON_V1_DIR = 'data_hitab_excl_v1'


def to_json_str(value):
    """Serialize dict/array cells (numpy arrays included) to JSON so CSV keeps them intact."""
    if isinstance(value, str):
        return value
    return json.dumps(value, ensure_ascii=False, default=lambda o: o.tolist() if hasattr(o, 'tolist') else str(o))


# --- IndoHiTab v1 (Indonesian): QA rows + one row per table ---
v1_json_cols = ['answer_coordinates', 'answer_text']
df_v1_qa = df_indohitab_raw.drop(columns=['table']).copy()
for col in v1_json_cols:
    df_v1_qa[col] = df_v1_qa[col].apply(to_json_str)

df_v1_tables = (
    df_indohitab_raw.drop_duplicates(subset='table_id')[['table_id', 'table_source', 'table']]
    .merge(df_indohitab[['table_id', 'qa_count']], on='table_id')
    .reset_index(drop=True)
)
df_v1_tables['table'] = df_v1_tables['table'].apply(to_json_str)

# --- HiTab (original) WITHOUT the v1 tables: tables (+content) and their QA rows ---
hitab_qa_cols = [c for c in df_hitab_raw.columns if c != 'table_content']
df_hitab_nonv1_tables = (
    df_indohitabv2_2.merge(df_hitab_raw.drop_duplicates(subset='table_id')[['table_id', 'table_content']], on='table_id')
    .reset_index(drop=True)
)
df_hitab_nonv1_qa = df_hitab_raw[df_hitab_raw['table_id'].isin(df_hitab_nonv1_tables['table_id'])][hitab_qa_cols].reset_index(drop=True)

# --- save each part to its own folder ---
os.makedirs(V1_DIR, exist_ok=True)
df_v1_tables.to_csv(os.path.join(V1_DIR, 'tables.csv'), index=False)
df_v1_qa.to_csv(os.path.join(V1_DIR, 'qa.csv'), index=False)
os.makedirs(HITAB_NON_V1_DIR, exist_ok=True)
df_hitab_nonv1_tables.to_csv(os.path.join(HITAB_NON_V1_DIR, 'tables.csv'), index=False)
df_hitab_nonv1_qa.to_csv(os.path.join(HITAB_NON_V1_DIR, 'qa.csv'), index=False)

# --- verification: the two parts must be disjoint and together make up the full HiTab ---
v1_ids = set(df_v1_tables['table_id'].astype(str))
rest_ids = set(df_hitab_nonv1_tables['table_id'].astype(str))
hitab_ids = set(df_hitab['table_id'].astype(str))
hitab_v1_qa_rows = df_hitab_raw['table_id'].astype(str).isin(v1_ids).sum()

checks = {
    'v1 and non-v1 tables are disjoint': not (v1_ids & rest_ids),
    'v1 + non-v1 tables == full HiTab tables': (v1_ids | rest_ids) == hitab_ids,
    'every v1 table exists in HiTab': v1_ids <= hitab_ids,
    'v1 QA count == HiTab QA count of the v1 tables': len(df_v1_qa) == hitab_v1_qa_rows,
    'v1 QA + non-v1 QA == full HiTab QA': len(df_v1_qa) + len(df_hitab_nonv1_qa) == len(df_hitab_raw),
}
print("Verification:")
for name, ok in checks.items():
    print(f"  [{'OK' if ok else 'FAIL'}] {name}")
assert all(checks.values())

print(f"\nIndoHiTab v1      -> ./{V1_DIR}/        : {len(df_v1_tables)} tables, {len(df_v1_qa)} QA")
print(f"HiTab without v1  -> ./{HITAB_NON_V1_DIR}/ : {len(df_hitab_nonv1_tables)} tables, {len(df_hitab_nonv1_qa)} QA")
print(pd.DataFrame({
    'indohitab_v1': df_v1_tables.groupby('table_source')['table_id'].nunique(),
    'hitab_without_v1': df_hitab_nonv1_tables.groupby('table_source')['table_id'].nunique(),
}).fillna(0).astype(int))

Verification:
  [OK] v1 and non-v1 tables are disjoint
  [OK] v1 + non-v1 tables == full HiTab tables
  [OK] every v1 table exists in HiTab
  [OK] v1 QA count == HiTab QA count of the v1 tables
  [OK] v1 QA + non-v1 QA == full HiTab QA

IndoHiTab v1      -> ./data_indohitab_v1/        : 600 tables, 2880 QA
HiTab without v1  -> ./data_hitab_excl_v1/ : 2997 tables, 7792 QA
              indohitab_v1  hitab_without_v1
table_source                                
nsf                      0               183
statcan                600               963
totto                    0              1851


## Sampling Scenarios (from IndoHiTab v2_2 pool)

Choose tables + QA pairs to sample from `df_indohitabv2_2` — the 2997 HiTab tables (`nsf` 183, `statcan` 963, `totto` 1851) not already covered by IndoHiTab v1.

- **Scenario 1 — Proportional:** targets `statcan` 1563, `nsf` 183, `totto` 1851 (the original full-HiTab distribution). Since only 963 `statcan` tables remain in the pool, this caps automatically and effectively takes the **entire v2_2 pool** (2997 tables) — i.e. "everything left, in its natural proportion."
- **Scenario 2 — Balanced (600 tables):** `nsf` 183 (its entire pool), `statcan` 209, `totto` 208 — an actual downsample of the pool.

Each scenario is sampled with a fixed seed and saved to its own folder with its own variables.

In [83]:
import os

SEED = 42
QA_COLS = ['id', 'table_id', 'table_source', 'sentence_id', 'sub_sentence_id', 'sub_sentence',
           'question', 'answer', 'aggregation', 'linked_cells', 'answer_formulas', 'reference_cells_map']

# v2_2 pool (table_id, table_source, qa_count) + table_content, joined back from the raw HiTab data
df_v2_2_pool = df_indohitabv2_2.merge(
    df_hitab_raw.drop_duplicates(subset='table_id')[['table_id', 'table_content']],
    on='table_id'
)

# All QA pairs belonging to tables in the v2_2 pool
df_v2_2_qa_raw = df_hitab_raw[df_hitab_raw['table_id'].isin(df_v2_2_pool['table_id'])]

print(f"v2_2 pool: {df_v2_2_pool['table_id'].nunique()} tables, {len(df_v2_2_qa_raw)} QA pairs")
print(df_v2_2_pool.groupby('table_source')['table_id'].nunique())


def sample_by_source(df_tables, target_counts, seed=SEED):
    """Sample table_ids per table_source according to target_counts (capped at pool size)."""
    parts = []
    for source, n in target_counts.items():
        pool = df_tables[df_tables['table_source'] == source]
        n_sample = min(n, len(pool))
        if n_sample < n:
            print(f"  [warn] {source}: requested {n}, only {len(pool)} available -> using {n_sample}")
        parts.append(pool.sample(n=n_sample, random_state=seed))
    return pd.concat(parts, ignore_index=True)


def build_scenario(df_tables, df_qa_raw, target_counts, folder, seed=SEED):
    """Sample tables + their QA pairs, save both to `folder`, and return the two DataFrames."""
    tables_sample = sample_by_source(df_tables, target_counts, seed=seed)
    qa_sample = df_qa_raw[df_qa_raw['table_id'].isin(tables_sample['table_id'])][QA_COLS].reset_index(drop=True)

    os.makedirs(folder, exist_ok=True)
    tables_sample.to_csv(os.path.join(folder, 'tables_sample.csv'), index=False)
    qa_sample.to_csv(os.path.join(folder, 'qa_sample.csv'), index=False)

    print(f"Saved scenario to ./{folder}/")
    print(tables_sample.groupby('table_source')['table_id'].nunique())
    print(f"Total tables: {tables_sample['table_id'].nunique()}, Total QA pairs: {len(qa_sample)}\n")

    return tables_sample, qa_sample

v2_2 pool: 2997 tables, 7792 QA pairs
table_source
nsf         183
statcan     963
totto      1851
Name: table_id, dtype: int64


In [84]:
# --- Scenario 1: Proportional (full original-HiTab ratio; caps to the whole v2_2 pool) ---
SCENARIO1_DIR = 'scenario1_proportional'
scenario1_targets = {'statcan': 261, 'nsf': 30, 'totto': 309}

df_tables_scenario1, df_qa_scenario1 = build_scenario(
    df_v2_2_pool, df_v2_2_qa_raw, scenario1_targets, SCENARIO1_DIR
)

Saved scenario to ./scenario1_proportional/
table_source
nsf         30
statcan    261
totto      309
Name: table_id, dtype: int64
Total tables: 600, Total QA pairs: 1838



In [85]:
# --- Scenario 2: Balanced (600 tables total; NSF fully used, STATCAN/OTTO downsampled ~evenly) ---
SCENARIO2_DIR = 'scenario2_balanced'
scenario2_targets = {'nsf': 183, 'statcan': 209, 'totto': 208}

df_tables_scenario2, df_qa_scenario2 = build_scenario(
    df_v2_2_pool, df_v2_2_qa_raw, scenario2_targets, SCENARIO2_DIR
)

Saved scenario to ./scenario2_balanced/
table_source
nsf        183
statcan    209
totto      208
Name: table_id, dtype: int64
Total tables: 600, Total QA pairs: 2096



In [86]:
# --- Comparison summary ---
summary = pd.DataFrame({
    'scenario1_proportional': df_tables_scenario1.groupby('table_source')['table_id'].nunique(),
    'scenario2_balanced': df_tables_scenario2.groupby('table_source')['table_id'].nunique(),
})
summary.loc['TOTAL'] = summary.sum()
print(summary)

              scenario1_proportional  scenario2_balanced
table_source                                            
nsf                               30                 183
statcan                          261                 209
totto                            309                 208
TOTAL                            600                 600


### QA totals & non-numeric data-cell analysis per scenario

For each scenario: sum the QA pairs per `table_source`, then parse each sampled table's `table_content` (`texts` grid + `top_header_rows_num` / `left_header_columns_num`) to count how many **data cells** — i.e. cells outside the header rows and header columns — are not numeric.

In [87]:
import ast


def parse_table_content(raw):
    """table_content is stored as a Python-literal string (single quotes) -> ast.literal_eval, not json."""
    if isinstance(raw, dict):
        return raw
    return ast.literal_eval(raw)


def is_numeric_cell(value):
    """A cell counts as numeric if it parses as a number after stripping common formatting (, % $)."""
    text = str(value).strip()
    if text == '':
        return False
    text = text.replace(',', '').replace('%', '').replace('$', '')
    try:
        float(text)
        return True
    except ValueError:
        return False


def count_non_numeric_cells(table_content_raw):
    """Count data cells (excluding header rows/columns) that are not numeric. Returns (total, non_numeric)."""
    content = parse_table_content(table_content_raw)
    texts = content['texts']
    top_h = content.get('top_header_rows_num', 0)
    left_h = content.get('left_header_columns_num', 0)

    total = non_numeric = 0
    for row in texts[top_h:]:
        for cell in row[left_h:]:
            total += 1
            if not is_numeric_cell(cell):
                non_numeric += 1
    return total, non_numeric


def analyze_non_numeric_cells(df_tables_sample):
    """Per-table breakdown of data-cell counts (excludes header rows/columns)."""
    counts = df_tables_sample['table_content'].apply(count_non_numeric_cells)
    out = df_tables_sample[['table_id', 'table_source']].copy()
    out['data_cells_total'] = [c[0] for c in counts]
    out['data_cells_non_numeric'] = [c[1] for c in counts]
    return out


def summarize_scenario(df_tables_sample, label):
    print(f"=== {label} ===")

    qa_sum = df_tables_sample.groupby('table_source')['qa_count'].sum()
    qa_sum.loc['TOTAL'] = qa_sum.sum()
    print("QA pairs (sum of qa_count) per source:")
    print(qa_sum)

    cell_stats = analyze_non_numeric_cells(df_tables_sample)
    by_source = cell_stats.groupby('table_source')[['data_cells_total', 'data_cells_non_numeric']].sum()
    by_source['pct_non_numeric'] = (by_source['data_cells_non_numeric'] / by_source['data_cells_total'] * 100).round(2)
    by_source.loc['TOTAL'] = [
        by_source['data_cells_total'].sum(),
        by_source['data_cells_non_numeric'].sum(),
        round(by_source['data_cells_non_numeric'].sum() / by_source['data_cells_total'].sum() * 100, 2),
    ]
    print("\nNon-numeric data cells (excluding header rows/columns) per source:")
    print(by_source)
    print()

    return cell_stats

In [88]:
cell_stats_scenario1 = summarize_scenario(df_tables_scenario1, 'Scenario 1 - Proportional')

=== Scenario 1 - Proportional ===
QA pairs (sum of qa_count) per source:
table_source
nsf         115
statcan    1396
totto       327
TOTAL      1838
Name: qa_count, dtype: int64

Non-numeric data cells (excluding header rows/columns) per source:
              data_cells_total  data_cells_non_numeric  pct_non_numeric
table_source                                                           
nsf                     4819.0                   324.0             6.72
statcan                29735.0                  4730.0            15.91
totto                  51840.0                 16150.0            31.15
TOTAL                  86394.0                 21204.0            24.54



In [89]:
cell_stats_scenario2 = summarize_scenario(df_tables_scenario2, 'Scenario 2 - Balanced')

=== Scenario 2 - Balanced ===
QA pairs (sum of qa_count) per source:
table_source
nsf         762
statcan    1114
totto       220
TOTAL      2096
Name: qa_count, dtype: int64

Non-numeric data cells (excluding header rows/columns) per source:
              data_cells_total  data_cells_non_numeric  pct_non_numeric
table_source                                                           
nsf                    26698.0                  2465.0             9.23
statcan                23871.0                  3497.0            14.65
totto                  33790.0                 11058.0            32.73
TOTAL                  84359.0                 17020.0            20.18



## Scenario 3 — Balanced mix targeting 2880 QA pairs (matches IndoHiTab v1's QA volume)

Explicitly drop any pool table with 0 QA pairs (a no-op here since `df_indohitabv2_2` is already filtered, but kept for safety), then build a balanced-mix scenario sized so its **QA total lands on 2880** — the same QA volume as IndoHiTab v1 — instead of capping at a fixed table count like Scenario 2.

`nsf` takes its entire remaining pool (183, same as Scenario 2's limiting source); `statcan`/`totto` table counts are searched (kept close to each other) until the sampled QA sum hits the 2880 target.

In [90]:
# Explicitly remove tables without QA pairs before sampling
df_v2_2_pool_valid = df_v2_2_pool[df_v2_2_pool['qa_count'] > 0].copy()
tables_dropped_v2_2_pool = len(df_v2_2_pool) - len(df_v2_2_pool_valid)
print(f"Dropped {tables_dropped_v2_2_pool} tables with 0 QA pairs; pool now {len(df_v2_2_pool_valid)} tables")


def find_balanced_target_counts(df_pool, target_qa, seed=SEED, search_range=600, refine_window=20):
    """Balanced mix: nsf takes its whole pool; statcan/totto counts are searched (kept close to
    each other) so the sampled QA sum lands as close as possible to target_qa."""
    nsf_pool_size = df_pool[df_pool['table_source'] == 'nsf']['table_id'].nunique()
    nsf_qa = sample_by_source(df_pool, {'nsf': nsf_pool_size}, seed=seed)['qa_count'].sum()
    remaining_qa = target_qa - nsf_qa

    # coarse pass: equal statcan/totto counts
    best = None
    for m in range(1, search_range):
        st_qa = sample_by_source(df_pool, {'statcan': m}, seed=seed)['qa_count'].sum()
        to_qa = sample_by_source(df_pool, {'totto': m}, seed=seed)['qa_count'].sum()
        diff = abs(st_qa + to_qa - remaining_qa)
        if best is None or diff < best[0]:
            best = (diff, m)
    base_m = best[1]

    # fine pass: let statcan/totto counts diverge slightly around base_m to close the gap exactly
    best2 = None
    for sm in range(max(1, base_m - refine_window), base_m + refine_window):
        for tm in range(max(1, base_m - refine_window), base_m + refine_window):
            st_qa = sample_by_source(df_pool, {'statcan': sm}, seed=seed)['qa_count'].sum()
            to_qa = sample_by_source(df_pool, {'totto': tm}, seed=seed)['qa_count'].sum()
            diff = abs(nsf_qa + st_qa + to_qa - target_qa)
            if best2 is None or diff < best2[0]:
                best2 = (diff, sm, tm)

    return {'nsf': nsf_pool_size, 'statcan': best2[1], 'totto': best2[2]}


scenario3_targets = find_balanced_target_counts(df_v2_2_pool_valid, target_qa=2880)
print("scenario3_targets:", scenario3_targets)

Dropped 0 tables with 0 QA pairs; pool now 2997 tables
scenario3_targets: {'nsf': 183, 'statcan': 339, 'totto': 356}


In [91]:
# --- Scenario 3: Balanced, sized to hit 2880 total QA pairs ---
SCENARIO3_DIR = 'scenario3_balanced_2880qa'

df_tables_scenario3, df_qa_scenario3 = build_scenario(
    df_v2_2_pool_valid, df_v2_2_qa_raw, scenario3_targets, SCENARIO3_DIR
)

Saved scenario to ./scenario3_balanced_2880qa/
table_source
nsf        183
statcan    339
totto      356
Name: table_id, dtype: int64
Total tables: 878, Total QA pairs: 2880



In [92]:
cell_stats_scenario3 = summarize_scenario(df_tables_scenario3, 'Scenario 3 - Balanced (2880 QA target)')

=== Scenario 3 - Balanced (2880 QA target) ===
QA pairs (sum of qa_count) per source:
table_source
nsf         762
statcan    1740
totto       378
TOTAL      2880
Name: qa_count, dtype: int64

Non-numeric data cells (excluding header rows/columns) per source:
              data_cells_total  data_cells_non_numeric  pct_non_numeric
table_source                                                           
nsf                    26698.0                  2465.0             9.23
statcan                39452.0                  6298.0            15.96
totto                  59858.0                 18385.0            30.71
TOTAL                 126008.0                 27148.0            21.54



## QA-balanced batches (8 per scenario)

Split each scenario's tables into 8 batches so every batch holds the same number of QA pairs. A table is indivisible, so tables are assigned greedily (largest `qa_count` first, into the currently lightest batch) and then refined by moves that shrink the gap; batch QA totals are equal whenever the scenario total divides evenly by 8, otherwise they differ by at most 1.

Result per scenario: `df_batches_scenarioN` with `table_id`, `table_source`, `qa_count`, `batch` (1-8), also saved as `batches.csv` in the scenario folder.

In [93]:
N_BATCHES = 8


def assign_qa_batches(df_tables, n_batches=N_BATCHES, seed=SEED):
    """Split tables into n_batches with QA totals as equal as possible (tables are indivisible)."""
    df = df_tables.sample(frac=1, random_state=seed).sort_values('qa_count', ascending=False, kind='stable')
    qa_by_table = dict(zip(df_tables['table_id'], df_tables['qa_count']))
    totals = [0] * n_batches
    assignment = {}

    for tid, qa in zip(df['table_id'], df['qa_count']):
        b = min(range(n_batches), key=totals.__getitem__)
        assignment[tid] = b
        totals[b] += qa

    # refinement: move a table from the fullest to the emptiest batch while that shrinks the gap
    improved = True
    while improved:
        improved = False
        hi = max(range(n_batches), key=totals.__getitem__)
        lo = min(range(n_batches), key=totals.__getitem__)
        gap = totals[hi] - totals[lo]
        for tid, b in assignment.items():
            qa = qa_by_table[tid]
            if b == hi and 0 < qa < gap:
                assignment[tid] = lo
                totals[hi] -= qa
                totals[lo] += qa
                improved = True
                break

    out = df_tables[['table_id', 'table_source', 'qa_count']].copy()
    out['batch'] = out['table_id'].map(assignment) + 1
    return out.sort_values(['batch', 'table_id']).reset_index(drop=True)


def batch_scenario(df_tables, folder, label):
    df_batches = assign_qa_batches(df_tables)
    df_batches.to_csv(os.path.join(folder, 'batches.csv'), index=False)

    print(f"=== {label}: {len(df_batches)} tables in {N_BATCHES} batches (saved to ./{folder}/batches.csv) ===")
    print(df_batches.groupby('batch').agg(tables=('table_id', 'count'), qa_pairs=('qa_count', 'sum')).T)
    print()
    return df_batches


df_batches_scenario1 = batch_scenario(df_tables_scenario1, SCENARIO1_DIR, 'Scenario 1 - Proportional')
df_batches_scenario2 = batch_scenario(df_tables_scenario2, SCENARIO2_DIR, 'Scenario 2 - Balanced')
df_batches_scenario3 = batch_scenario(df_tables_scenario3, SCENARIO3_DIR, 'Scenario 3 - Balanced (2880 QA target)')

=== Scenario 1 - Proportional: 600 tables in 8 batches (saved to ./scenario1_proportional/batches.csv) ===
batch       1    2    3    4    5    6    7    8
tables     74   75   75   75   75   76   75   75
qa_pairs  230  230  230  230  230  230  229  229

=== Scenario 2 - Balanced: 600 tables in 8 batches (saved to ./scenario2_balanced/batches.csv) ===
batch       1    2    3    4    5    6    7    8
tables     74   75   75   75   75   75   75   76
qa_pairs  262  262  262  262  262  262  262  262

=== Scenario 3 - Balanced (2880 QA target): 878 tables in 8 batches (saved to ./scenario3_balanced_2880qa/batches.csv) ===
batch       1    2    3    4    5    6    7    8
tables    108  110  110  110  110  110  110  110
qa_pairs  360  360  360  360  360  360  360  360



## QA inclusion check & QA-type analysis

1. **Inclusion check** — for every scenario, assert that all QA pairs belonging to the chosen tables are in `df_qa_scenarioN` (none missing, none from unchosen tables, no duplicates, per-table counts match `qa_count`).
2. **QA type** — HiTab's `aggregation` column is the QA type (a list such as `['none']`, `['pair-argmax']`, `['sum', 'div']`). Each QA is classified as `lookup (none)` (direct cell lookup), `single-op` or `multi-op` (several chained operations), and operation frequencies are counted per source. Results are returned as `df_qa_types_scenarioN` (QA rows + `agg_ops`, `n_ops`, `qa_type`).

In [94]:
def check_qa_included(df_tables, df_qa, df_qa_source, label):
    """Assert that every QA pair of the chosen tables is present in df_qa (and nothing else)."""
    expected = df_qa_source[df_qa_source['table_id'].isin(df_tables['table_id'])]
    per_table_actual = df_qa.groupby('table_id').size()

    checks = {
        'every chosen table has QA rows': set(df_tables['table_id']) == set(df_qa['table_id']),
        'no QA from unchosen tables': set(df_qa['table_id']).issubset(set(df_tables['table_id'])),
        'all source QA ids present (none missing)': set(expected['id']) == set(df_qa['id']),
        'QA ids unique (no duplicates)': df_qa['id'].is_unique,
        'per-table QA count == qa_count': per_table_actual.reindex(df_tables['table_id']).values.tolist()
                                          == df_tables['qa_count'].tolist(),
        'QA rows == sum(qa_count)': len(df_qa) == df_tables['qa_count'].sum(),
    }
    print(f"=== QA inclusion check: {label} ===")
    for name, ok in checks.items():
        print(f"  [{'OK' if ok else 'FAIL'}] {name}")
    print(f"  tables={len(df_tables)}, QA rows={len(df_qa)}, sum(qa_count)={df_tables['qa_count'].sum()}\n")
    assert all(checks.values()), f"QA inclusion check failed for {label}"


def parse_aggregation(raw):
    """`aggregation` is stored as a Python-literal string, e.g. "['sum', 'div']"."""
    return list(raw) if isinstance(raw, (list, tuple)) else ast.literal_eval(raw)


def analyze_qa_types(df_qa, label):
    df = df_qa.copy()
    df['agg_ops'] = df['aggregation'].apply(parse_aggregation)
    df['n_ops'] = df['agg_ops'].apply(len)
    df['qa_type'] = df.apply(
        lambda r: 'lookup (none)' if r['agg_ops'] == ['none'] else ('single-op' if r['n_ops'] == 1 else 'multi-op'),
        axis=1,
    )

    print(f"=== QA types: {label} ===")
    print("QA type by source (count):")
    print(pd.crosstab(df['table_source'], df['qa_type'], margins=True, margins_name='TOTAL'))
    print("\nQA type by source (% of the source's QA):")
    print((pd.crosstab(df['table_source'], df['qa_type'], normalize='index') * 100).round(1))

    ops = df.explode('agg_ops')
    ops_by_source = pd.crosstab(ops['agg_ops'], ops['table_source'])
    ops_by_source['TOTAL'] = ops_by_source.sum(axis=1)
    ops_by_source = ops_by_source.sort_values('TOTAL', ascending=False)
    ops_by_source.loc['TOTAL'] = ops_by_source.sum()
    print("\nAggregation operations (a multi-op QA counts once per operation):")
    print(ops_by_source)
    print()
    return df

In [95]:
scenarios = {
    'Scenario 1 - Proportional': (df_tables_scenario1, df_qa_scenario1),
    'Scenario 2 - Balanced': (df_tables_scenario2, df_qa_scenario2),
    'Scenario 3 - Balanced (2880 QA target)': (df_tables_scenario3, df_qa_scenario3),
}

# 1. every QA pair of the chosen tables must be included
for label, (df_t, df_q) in scenarios.items():
    check_qa_included(df_t, df_q, df_v2_2_qa_raw, label)

# 2. QA type analysis
df_qa_types_scenario1 = analyze_qa_types(df_qa_scenario1, 'Scenario 1 - Proportional')
df_qa_types_scenario2 = analyze_qa_types(df_qa_scenario2, 'Scenario 2 - Balanced')
df_qa_types_scenario3 = analyze_qa_types(df_qa_scenario3, 'Scenario 3 - Balanced (2880 QA target)')

# 3. side-by-side: QA type share (%) per scenario
qa_type_compare = pd.DataFrame({
    'scenario1_proportional': df_qa_types_scenario1['qa_type'].value_counts(normalize=True),
    'scenario2_balanced': df_qa_types_scenario2['qa_type'].value_counts(normalize=True),
    'scenario3_balanced_2880qa': df_qa_types_scenario3['qa_type'].value_counts(normalize=True),
}).mul(100).round(1)
print("QA type share (%) per scenario:")
print(qa_type_compare.to_string())

=== QA inclusion check: Scenario 1 - Proportional ===
  [OK] every chosen table has QA rows
  [OK] no QA from unchosen tables
  [OK] all source QA ids present (none missing)
  [OK] QA ids unique (no duplicates)
  [OK] per-table QA count == qa_count
  [OK] QA rows == sum(qa_count)
  tables=600, QA rows=1838, sum(qa_count)=1838

=== QA inclusion check: Scenario 2 - Balanced ===
  [OK] every chosen table has QA rows
  [OK] no QA from unchosen tables
  [OK] all source QA ids present (none missing)
  [OK] QA ids unique (no duplicates)
  [OK] per-table QA count == qa_count
  [OK] QA rows == sum(qa_count)
  tables=600, QA rows=2096, sum(qa_count)=2096

=== QA inclusion check: Scenario 3 - Balanced (2880 QA target) ===
  [OK] every chosen table has QA rows
  [OK] no QA from unchosen tables
  [OK] all source QA ids present (none missing)
  [OK] QA ids unique (no duplicates)
  [OK] per-table QA count == qa_count
  [OK] QA rows == sum(qa_count)
  tables=878, QA rows=2880, sum(qa_count)=2880

=== 

## Pilot test set (Scenario 3)

Every batch contains its own pilot: the **same number of pilot tables in each of the 8 batches** (`PILOT_PER_BATCH`, default 1 → 8 tables), with the total QA pairs as close to ~50 as possible.

Together the pilot tables cover **every QA type in Scenario 3**: all `aggregation` operations (`none`, `div`, `pair-argmax`, ... `topk-argmin`) plus the three categories `lookup (none)`, `single-op`, `multi-op`. Rare types can't appear in every batch (e.g. `topk-argmin` occurs once in the whole scenario), so coverage is required over the pilot as a whole.

Found with a seeded randomized hill-climb (coverage is weighted far above the QA-count gap). The result is flagged in `df_batches_scenario3['is_pilot']` (re-saved to `batches.csv`), and the pilot rows are also returned as `df_pilot_tables_scenario3` / `df_pilot_qa_scenario3` and saved as `pilot_tables.csv` / `pilot_qa.csv` in the scenario 3 folder. Set `PILOT_PER_BATCH = 2` for 16 tables (~50 QA still reachable).

In [96]:
import random

PILOT_PER_BATCH = 1
PILOT_TARGET_QA = 50


def select_pilot_tables(df_batches, df_qa_types, per_batch=PILOT_PER_BATCH, target_qa=PILOT_TARGET_QA,
                        seed=SEED, n_restarts=300, n_iters=4000):
    """Pick `per_batch` tables from EVERY batch, ~target_qa QA pairs in total, covering every QA type."""
    ops = df_qa_types.explode('agg_ops').groupby('table_id')['agg_ops'].agg(set)
    cats = df_qa_types.groupby('table_id')['qa_type'].agg(set)
    info = df_batches.set_index('table_id')
    types = {t: ops[t] | cats[t] for t in info.index}
    qa = info['qa_count'].to_dict()
    batch_of = info['batch'].to_dict()
    required = set().union(*types.values())
    by_batch = info.reset_index().groupby('batch')['table_id'].apply(list).to_dict()
    rng = random.Random(seed)

    def score(sel):
        covered = set().union(*(types[t] for t in sel))
        return 1000 * len(required - covered) + abs(sum(qa[t] for t in sel) - target_qa)

    best = None
    for _ in range(n_restarts):
        sel = []
        for b in sorted(by_batch):
            sel += rng.sample(by_batch[b], per_batch)
        cur = score(sel)

        for _ in range(n_iters):
            if cur == 0:
                break
            i = rng.randrange(len(sel))
            cand = [t for t in by_batch[batch_of[sel[i]]] if t not in sel]  # swap within the same batch
            if not cand:
                continue
            new = sel[:]
            new[i] = rng.choice(cand)
            s = score(new)
            if s <= cur:
                sel, cur = new, s

        if best is None or cur < best[0]:
            best = (cur, sel[:])
        if best[0] == 0:
            break

    return best[1], required


pilot_ids, required_types = select_pilot_tables(df_batches_scenario3, df_qa_types_scenario3)

# flag in the batches table + save
df_batches_scenario3['is_pilot'] = df_batches_scenario3['table_id'].isin(pilot_ids)
df_batches_scenario3.to_csv(os.path.join(SCENARIO3_DIR, 'batches.csv'), index=False)

df_pilot_tables_scenario3 = df_batches_scenario3[df_batches_scenario3['is_pilot']].reset_index(drop=True)
df_pilot_qa_scenario3 = df_qa_types_scenario3[df_qa_types_scenario3['table_id'].isin(pilot_ids)].reset_index(drop=True)
df_pilot_tables_scenario3.to_csv(os.path.join(SCENARIO3_DIR, 'pilot_tables.csv'), index=False)
df_pilot_qa_scenario3.drop(columns=['agg_ops', 'n_ops']).to_csv(os.path.join(SCENARIO3_DIR, 'pilot_qa.csv'), index=False)

# verification
pilot_types = set(df_pilot_qa_scenario3.explode('agg_ops')['agg_ops']) | set(df_pilot_qa_scenario3['qa_type'])
missing_types = required_types - pilot_types
pilot_tables_per_batch = df_pilot_tables_scenario3['batch'].value_counts().reindex(sorted(df_batches_scenario3['batch'].unique()), fill_value=0)
assert (pilot_tables_per_batch == PILOT_PER_BATCH).all(), "every batch must contain exactly PILOT_PER_BATCH pilot tables"
assert not missing_types, f"pilot misses QA types: {sorted(missing_types)}"

print(f"Pilot: {len(df_pilot_tables_scenario3)} tables, {len(df_pilot_qa_scenario3)} QA pairs "
      f"(saved to ./{SCENARIO3_DIR}/pilot_tables.csv, pilot_qa.csv)")
print(f"QA types covered: {len(pilot_types & required_types)}/{len(required_types)}  "
      f"missing: {sorted(missing_types) or 'none'}")
print("\nPilot per batch:")
print(df_pilot_tables_scenario3.groupby('batch').agg(
    pilot_tables=('table_id', 'count'), pilot_qa=('qa_count', 'sum'), table_ids=('table_id', list)).to_string())
print("\nTables per source:", df_pilot_tables_scenario3['table_source'].value_counts().to_dict())
print("QA per type category:", df_pilot_qa_scenario3['qa_type'].value_counts().to_dict())
print("\nOperation counts in the pilot:")
print(df_pilot_qa_scenario3.explode('agg_ops')['agg_ops'].value_counts().to_string())
print()
print(df_pilot_tables_scenario3.to_string(index=False))

Pilot: 8 tables, 52 QA pairs (saved to ./scenario3_balanced_2880qa/pilot_tables.csv, pilot_qa.csv)
QA types covered: 22/22  missing: none

Pilot per batch:
       pilot_tables  pilot_qa           table_ids
batch                                            
1                 1        13               [108]
2                 1         6               [675]
3                 1         6              [1863]
4                 1        11               [489]
5                 1        11              [2412]
6                 1         2              [1118]
7                 1         2              [1005]
8                 1         1  [415_totto48055-2]

Tables per source: {'statcan': 7, 'totto': 1}
QA per type category: {'single-op': 26, 'lookup (none)': 19, 'multi-op': 7}

Operation counts in the pilot:
agg_ops
none            19
pair-argmax      9
sum              6
greater_than     5
diff             2
less_than        2
pair-argmin      2
opposite         2
argmin           2
range     

## Overlap check with IndoHiTab v1

Make sure no scenario contains a table (or QA) that is already in IndoHiTab v1. For every scenario, the `table_id`s of each DataFrame (tables, QA, batches, pilot) **and of each saved CSV** are compared with the v1 `table_id`s; the QA `id`s are compared with the v1 QA `id`s. The cell asserts that every overlap count is 0.

In [97]:
v1_table_ids = {str(t) for t in df_indohitab_raw['table_id']}
v1_qa_ids = {str(i) for i in df_indohitab_raw['id']}


def table_ids_of(df):
    return {str(t) for t in df['table_id']}


def check_no_overlap_with_v1(label, variables, saved_files):
    """variables: {name: DataFrame}; saved_files: {name: csv path}. Every table_id must be absent from IndoHiTab v1."""
    rows = []
    for name, df in variables.items():
        ids = table_ids_of(df)
        rows.append((name, 'variable', len(ids), len(ids & v1_table_ids)))
    for name, path in saved_files.items():
        ids = set(pd.read_csv(path, usecols=['table_id'], dtype=str)['table_id'])
        rows.append((name, 'saved csv', len(ids), len(ids & v1_table_ids)))
    df_out = pd.DataFrame(rows, columns=['data', 'kind', 'unique_tables', 'overlap_with_v1'])
    print(f"=== Overlap with IndoHiTab v1: {label} ===")
    print(df_out.to_string(index=False))
    print()
    return df_out


overlap_reports = []
for n, (df_t, df_q, folder) in {
    1: (df_tables_scenario1, df_qa_scenario1, SCENARIO1_DIR),
    2: (df_tables_scenario2, df_qa_scenario2, SCENARIO2_DIR),
    3: (df_tables_scenario3, df_qa_scenario3, SCENARIO3_DIR),
}.items():
    variables = {f'df_tables_scenario{n}': df_t, f'df_qa_scenario{n}': df_q, f'df_batches_scenario{n}': globals().get(f'df_batches_scenario{n}')}
    files = {f: os.path.join(folder, f) for f in ['tables_sample.csv', 'qa_sample.csv', 'batches.csv']}
    if n == 3:
        variables.update({'df_pilot_tables_scenario3': df_pilot_tables_scenario3, 'df_pilot_qa_scenario3': df_pilot_qa_scenario3})
        files.update({f: os.path.join(folder, f) for f in ['pilot_tables.csv', 'pilot_qa.csv']})
    overlap_reports.append(check_no_overlap_with_v1(f'Scenario {n}', variables, files))

# QA ids must not coincide with v1 QA ids either
qa_id_overlap = {n: len({str(i) for i in df_q['id']} & v1_qa_ids)
                 for n, df_q in {1: df_qa_scenario1, 2: df_qa_scenario2, 3: df_qa_scenario3}.items()}
print("QA id overlap with v1 (per scenario):", qa_id_overlap)

total_overlap = sum(r['overlap_with_v1'].sum() for r in overlap_reports) + sum(qa_id_overlap.values())
assert total_overlap == 0, f"Found {total_overlap} overlapping table/QA ids with IndoHiTab v1"
print("\nOK: no table or QA in any scenario (variables and saved files) overlaps with IndoHiTab v1.")

=== Overlap with IndoHiTab v1: Scenario 1 ===
                data      kind  unique_tables  overlap_with_v1
 df_tables_scenario1  variable            600                0
     df_qa_scenario1  variable            600                0
df_batches_scenario1  variable            600                0
   tables_sample.csv saved csv            600                0
       qa_sample.csv saved csv            600                0
         batches.csv saved csv            600                0

=== Overlap with IndoHiTab v1: Scenario 2 ===
                data      kind  unique_tables  overlap_with_v1
 df_tables_scenario2  variable            600                0
     df_qa_scenario2  variable            600                0
df_batches_scenario2  variable            600                0
   tables_sample.csv saved csv            600                0
       qa_sample.csv saved csv            600                0
         batches.csv saved csv            600                0

=== Overlap with IndoHiT